# CBSE Question Paper Generator — Backend Prototype

**What this notebook does**

1. You describe / configure a paper (class, subject, section structure, marks, difficulty).
2. Gemini generates the questions, section by section.
3. A web search step (Tavily) pulls reference context so questions aren't just invented from thin air.
4. Any question that needs a figure gets a **TikZ diagram** — generated by Gemini and **self-checked by actually compiling it**; if it fails to compile, the error is fed back to Gemini and it retries automatically. That's what removes the "go check the diagram yourself" step.
5. Everything is assembled into one CBSE-formatted LaTeX document, compiled to PDF, and also exported as `.tex` and `.txt`.
6. A separate `edit_question()` function lets you say *"change Q5"* in plain English — it rewrites only that block and recompiles, instead of regenerating the whole paper.

**Important — read before running**

This notebook was built and *mechanically tested* (LaTeX/TikZ compilation, file export) inside a sandboxed environment that has **no access to the Gemini or Tavily APIs** — only PyPI/GitHub-type domains are reachable there. So:

- Every function that touches LaTeX/PDF/TXT generation has already been tested end-to-end and works (see the "Offline pipeline self-test" section near the end — it actually ran, with real output).
- Every function that calls Gemini or Tavily is written and ready, but **you need to run this notebook yourself** (locally, or in Google Colab) with your own API keys for those parts to execute.

**Free-tier keys you'll need**
- Gemini: [aistudio.google.com](https://aistudio.google.com/) → "Get API key" (free tier available; check current rate limits).
- Tavily (web search): [tavily.com](https://tavily.com/) → free tier (~1,000 searches/month). Optional — the paper still generates without it, just without the reference-search grounding.

**Why not MCP for the search step?** MCP is great when an *agent runtime* (Claude Desktop, Claude Code, etc.) needs to plug into external tools. Here you just need one Python function that calls a search API and returns text — a direct Tavily/SerpAPI call via LangChain's tool wrapper is simpler, has no extra moving parts, and costs nothing extra. MCP would be worth it later if you turn this into a persistent multi-tool agent.


## 1. Install dependencies

In [1]:
# Run this once. Safe to re-run (pip will just confirm things are already installed).
# The '|| ... --break-system-packages' fallback handles environments (e.g. some Linux setups)
# that block system-wide pip installs by default; Colab/most local setups won't need it.
!pip install --quiet google-generativeai langchain langchain-google-genai langgraph langchain-community tavily-python nbformat pypdf \
  || pip install --quiet --break-system-packages google-generativeai langchain langchain-google-genai langgraph langchain-community tavily-python nbformat pypdf
print("Dependencies installed.")

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Dependencies installed.


## 2. API keys
Set these as environment variables before running, **or** run this notebook interactively and uncomment the `getpass` lines to type them in securely (do NOT hardcode keys in the notebook).

In [2]:
import os

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY", "")

# If running interactively and the env vars above are empty, uncomment:
# import getpass
# GEMINI_API_KEY = GEMINI_API_KEY or getpass.getpass("Enter your Gemini API key: ")
# TAVILY_API_KEY = TAVILY_API_KEY or getpass.getpass("Enter your Tavily API key (optional, press Enter to skip): ")

if not GEMINI_API_KEY:
    print("\u26a0\ufe0f  GEMINI_API_KEY not set. The question-generation and diagram steps will be skipped until you set it.")
if not TAVILY_API_KEY:
    print("\u2139\ufe0f  TAVILY_API_KEY not set. Web-search grounding will be skipped (paper still generates from Gemini's own knowledge).")

⚠️  GEMINI_API_KEY not set. The question-generation and diagram steps will be skipped until you set it.
ℹ️  TAVILY_API_KEY not set. Web-search grounding will be skipped (paper still generates from Gemini's own knowledge).


## 3. Paper configuration

This is a **placeholder config** shaped like a real CBSE Class 10 Maths paper (5 sections, MCQ → Case Study, standard marks split). Replace every field with your actual rubric once you send it over — the rest of the notebook reads entirely from this dict, nothing else needs to change.

In [3]:
CONFIG = {
    "board": "CBSE",
    "class_name": "10",
    "subject": "Mathematics",
    "duration_minutes": 180,
    "max_marks": 80,
    # "gemini-3.1-pro-preview" is used as a sensible default — check https://ai.google.dev/gemini-api/docs/models
    # for whatever the current best/latest Gemini model is and update this if needed.
    "model_name": "gemini-3.1-pro-preview",
    "topics": [
        "Real Numbers", "Polynomials", "Quadratic Equations", "Triangles",
        "Coordinate Geometry", "Trigonometry", "Circles", "Statistics", "Probability",
    ],
    "sections": [
        {"name": "Section A", "question_type": "MCQ",           "num_questions": 4, "marks_each": 1, "difficulty": "easy"},
        {"name": "Section B", "question_type": "Short Answer",  "num_questions": 3, "marks_each": 2, "difficulty": "medium"},
        {"name": "Section C", "question_type": "Short Answer",  "num_questions": 2, "marks_each": 3, "difficulty": "medium"},
        {"name": "Section D", "question_type": "Long Answer",   "num_questions": 2, "marks_each": 5, "difficulty": "hard"},
        {"name": "Section E", "question_type": "Case Study",    "num_questions": 1, "marks_each": 4, "difficulty": "mixed"},
    ],
}

OUTPUT_DIR = "/content/cbse_tool_output"  # change if not using Colab
BUILD_DIR = "/content/cbse_tool_build"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BUILD_DIR, exist_ok=True)
print("Config loaded:", CONFIG["subject"], "- Class", CONFIG["class_name"])

Config loaded: Mathematics - Class 10


## 4. Initialise the Gemini model (via LangChain)

In [4]:
GEMINI_MODEL = None
if GEMINI_API_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    GEMINI_MODEL = ChatGoogleGenerativeAI(
        model=CONFIG["model_name"],
        google_api_key=GEMINI_API_KEY,
        temperature=0.4,
    )
    print("Gemini model initialised:", CONFIG["model_name"])
else:
    print("Gemini model NOT initialised (no API key). Set GEMINI_API_KEY above and re-run this cell.")

Gemini model NOT initialised (no API key). Set GEMINI_API_KEY above and re-run this cell.


## 5. LaTeX template + compile helpers

CBSE-style header/footer, and a generic `_compile_string()` helper that writes a `.tex` file and runs `pdflatex` on it, returning success + the full compiler log (used both for the final paper and for validating each diagram in isolation).

In [5]:
import subprocess, os

def build_latex_header(config):
    hours = config["duration_minutes"] // 60
    minutes = config["duration_minutes"] % 60
    time_str = f"{hours} Hours" + (f" {minutes} Minutes" if minutes else "")
    return r"""\documentclass[12pt,a4paper]{article}
\usepackage[margin=0.9in]{geometry}
\usepackage{amsmath,amssymb}
\usepackage{tikz}
\usepackage{enumitem}
\usepackage{fancyhdr}
\setlength{\headheight}{15pt}
\pagestyle{fancy}
\fancyhf{}
\lhead{""" + config["board"] + r"""}
\rhead{Class """ + config["class_name"] + r""" -- """ + config["subject"] + r"""}
\cfoot{\thepage}

\begin{document}
\begin{center}
{\Large \textbf{""" + config["subject"].upper() + r"""}}\\[4pt]
Time Allowed: """ + time_str + r""" \hfill Maximum Marks: """ + str(config["max_marks"]) + r"""
\end{center}
\hrule
\vspace{8pt}
\textit{General Instructions: This question paper follows the """ + config["board"] + r""" format guidelines.}
\vspace{6pt}
"""

LATEX_FOOTER = r"\end{document}"

LATEX_DIAGRAM_TEST_WRAPPER = r"""\documentclass{standalone}
\usepackage{tikz}
\begin{document}
\begin{tikzpicture}
%s
\end{tikzpicture}
\end{document}
"""

def _compile_string(latex_source, base_name, workdir):
    """Writes latex_source to <workdir>/<base_name>.tex and compiles it with pdflatex.
    Returns (success: bool, full_log: str)."""
    os.makedirs(workdir, exist_ok=True)
    tex_path = os.path.join(workdir, base_name + ".tex")
    with open(tex_path, "w") as f:
        f.write(latex_source)
    try:
        result = subprocess.run(
            ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", base_name + ".tex"],
            cwd=workdir, capture_output=True, text=True, timeout=60,
        )
        log = result.stdout + result.stderr
        pdf_path = os.path.join(workdir, base_name + ".pdf")
        success = os.path.exists(pdf_path) and result.returncode == 0
        return success, log
    except FileNotFoundError:
        return False, (
            "pdflatex was not found on PATH. You need a LaTeX distribution installed:\\n"
            "  Windows: MiKTeX -> https://miktex.org/download\\n"
            "  macOS:   MacTeX -> https://tug.org/mactex/\\n"
            "  Linux:   sudo apt-get install texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-pictures lmodern\\n"
            "After installing, restart your terminal/VS Code/Jupyter kernel so PATH updates, then re-run this cell."
        )
    except subprocess.TimeoutExpired as e:
        return False, f"pdflatex timed out: {e}"

print("LaTeX template + compile helpers ready.")

LaTeX template + compile helpers ready.


### 5b. Pre-flight check — is LaTeX actually installed?
Run this before anything else touches `pdflatex`. Saves you from hitting a `FileNotFoundError` deep inside a compile call.

In [6]:
import shutil

_pdflatex_path = shutil.which("pdflatex")
if _pdflatex_path is None:
    print("\u26a0\ufe0f  pdflatex not found on PATH. Install a LaTeX distribution first:")
    print("   Windows: MiKTeX -> https://miktex.org/download")
    print("   macOS:   MacTeX -> https://tug.org/mactex/")
    print("   Linux:   sudo apt-get install texlive-latex-base texlive-latex-extra texlive-fonts-recommended texlive-pictures lmodern")
    print("Then RESTART your terminal / VS Code / Jupyter kernel (PATH only updates for new sessions) and re-run this cell.")
else:
    print("pdflatex found at:", _pdflatex_path)

pdflatex found at: /usr/bin/pdflatex


## 6. Diagram generation — self-checking TikZ

This is the important bit for "the diagram has to be the finest, I shouldn't have to check it":

1. Ask Gemini for TikZ code for the described figure.
2. Compile *that snippet alone* in an isolated `standalone` document.
3. If it fails, send the exact compiler error back to Gemini and ask it to fix it — up to `max_attempts` times.
4. Only a diagram that actually compiles cleanly gets inserted into the real paper.

This won't guarantee the diagram is pedagogically perfect, but it guarantees it will never be a broken/garbled figure in the final PDF — which is the failure mode manual TikZ generation usually has.

In [7]:
def generate_tikz_diagram(description, model, max_attempts=3, build_dir="/tmp/diagram_test"):
    if model is None:
        raise RuntimeError("Gemini model not initialised - set GEMINI_API_KEY first.")

    prompt = f"""Write ONLY the TikZ drawing commands (the contents that go BETWEEN
\\begin{{tikzpicture}} and \\end{{tikzpicture}} - do not include those two lines themselves)
for the following diagram, to be used in a CBSE exam question:

Diagram needed: {description}

Rules:
- Output ONLY valid TikZ drawing commands, nothing else (no markdown fences, no explanation).
- Label vertices/angles/lengths clearly, using standard exam-diagram conventions
  (e.g. right-angle marks, tick marks for equal sides).
- Keep it visually clean and appropriately sized for a printed exam paper."""

    tikz_body = model.invoke(prompt).content.strip()
    tikz_body = tikz_body.replace("```latex", "").replace("```tikz", "").replace("```", "").strip()

    for attempt in range(1, max_attempts + 1):
        test_doc = LATEX_DIAGRAM_TEST_WRAPPER % tikz_body
        ok, log = _compile_string(test_doc, f"diagram_test_{abs(hash(description)) % 100000}", build_dir)
        if ok:
            return tikz_body

        fix_prompt = f"""This TikZ code failed to compile.

COMPILER ERROR (tail):
{log[-1500:]}

TIKZ CODE:
{tikz_body}

Return ONLY the corrected TikZ body (same rules as before: no wrapper, no fences, no explanation)."""
        tikz_body = model.invoke(fix_prompt).content.strip()
        tikz_body = tikz_body.replace("```latex", "").replace("```tikz", "").replace("```", "").strip()

    raise RuntimeError(f"Diagram did not compile after {max_attempts} attempts: {description}")

print("Diagram generator ready.")

Diagram generator ready.

## 7. Web search grounding (Tavily)

In [8]:
def search_reference_questions(topic, num_results=3):
    """Returns a list of short text snippets about `topic` to ground question
    generation. Skips gracefully (returns []) if no Tavily key is set."""
    if not TAVILY_API_KEY:
        return []
    from tavily import TavilyClient
    client = TavilyClient(api_key=TAVILY_API_KEY)
    resp = client.search(query=f"CBSE exam sample questions {topic}", max_results=num_results)
    return [r.get("content", "") for r in resp.get("results", [])]

print("Search helper ready.")

Search helper ready.


## 8. Question generation

In [9]:
import json, re

def _extract_json(text):
    """Strips markdown fences and parses the first JSON array/object found."""
    cleaned = re.sub(r"```json|```", "", text).strip()
    return json.loads(cleaned)

def generate_section_questions(section, config, model, search_context=""):
    """Returns a list of dicts:
    {"text": str, "needs_diagram": bool, "diagram_description": str or None}"""
    if model is None:
        raise RuntimeError("Gemini model not initialised - set GEMINI_API_KEY first.")

    prompt = f"""You are setting the "{section['name']}" section of a {config['board']} Class
{config['class_name']} {config['subject']} exam paper.

Section details:
- Question type: {section['question_type']}
- Number of questions: {section['num_questions']}
- Marks per question: {section['marks_each']}
- Difficulty: {section['difficulty']}
- Topics to draw from: {', '.join(config['topics'])}

Reference context (may be empty):
{search_context[:2000]}

Return a JSON array of exactly {section['num_questions']} objects, each with keys:
- "text": the full question text (plain LaTeX-safe text, no diagram description inside it)
- "needs_diagram": true/false
- "diagram_description": if needs_diagram is true, a precise description of the figure
  needed (shapes, labels, given measurements); otherwise null

Return ONLY the JSON array, nothing else."""

    raw = model.invoke(prompt).content
    return _extract_json(raw)

print("Question generator ready.")

Question generator ready.


## 9. LangGraph orchestration

Wires the steps above into a small graph: **generate sections → assemble LaTeX → compile → (if it fails) ask Gemini to fix the whole document → recompile**, bounded to a couple of retries. Defining the graph doesn't call any API — it's safe to run this cell even without keys set.

In [10]:
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END

class PaperState(TypedDict):
    config: Dict[str, Any]
    sections_data: List[Dict[str, Any]]
    latex_body: str
    compile_ok: bool
    compile_log: str
    retries: int

def node_generate_sections(state: PaperState):
    config = state["config"]
    sections_data = []
    for section in config["sections"]:
        ctx_snippets = []
        for topic in config["topics"][:2]:  # cap search calls per section
            ctx_snippets.extend(search_reference_questions(topic))
        questions = generate_section_questions(section, config, GEMINI_MODEL, "\n".join(ctx_snippets))
        sections_data.append({"section": section, "questions": questions})
    return {"sections_data": sections_data}

def node_assemble_latex(state: PaperState):
    config = state["config"]
    body = build_latex_header(config)
    q_num = 1
    for sd in state["sections_data"]:
        section = sd["section"]
        body += f"\n\\section*{{{section['name']}}}\n"
        body += f"\\textit{{({section['question_type']}, {section['marks_each']} marks each)}}\n"
        body += "\\begin{enumerate}\n"
        for q in sd["questions"]:
            body += f"% %%% Q{q_num}_START\n"
            body += f"\\item {q['text']}\n"
            if q.get("needs_diagram"):
                tikz = generate_tikz_diagram(q["diagram_description"], GEMINI_MODEL)
                body += f"\\begin{{center}}\n\\begin{{tikzpicture}}\n{tikz}\n\\end{{tikzpicture}}\n\\end{{center}}\n"
            body += f"% %%% Q{q_num}_END\n"
            q_num += 1
        body += "\\end{enumerate}\n"
    body += LATEX_FOOTER
    return {"latex_body": body}

def node_compile(state: PaperState):
    ok, log = _compile_string(state["latex_body"], "generated_paper", BUILD_DIR)
    return {"compile_ok": ok, "compile_log": log}

def node_fix_errors(state: PaperState):
    fix_prompt = f"""This full CBSE exam LaTeX document failed to compile.

COMPILER ERROR (tail):
{state['compile_log'][-2000:]}

FULL LATEX SOURCE:
{state['latex_body']}

Return ONLY the corrected, complete LaTeX document (from \\documentclass to \\end{{document}})."""
    fixed = GEMINI_MODEL.invoke(fix_prompt).content.strip()
    fixed = fixed.replace("```latex", "").replace("```", "").strip()
    return {"latex_body": fixed, "retries": state.get("retries", 0) + 1}

def route_after_compile(state: PaperState):
    if state["compile_ok"] or state.get("retries", 0) >= 2:
        return END
    return "fix_errors"

graph = StateGraph(PaperState)
graph.add_node("generate_sections", node_generate_sections)
graph.add_node("assemble_latex", node_assemble_latex)
graph.add_node("compile", node_compile)
graph.add_node("fix_errors", node_fix_errors)
graph.set_entry_point("generate_sections")
graph.add_edge("generate_sections", "assemble_latex")
graph.add_edge("assemble_latex", "compile")
graph.add_conditional_edges("compile", route_after_compile, {"fix_errors": "fix_errors", END: END})
graph.add_edge("fix_errors", "compile")
paper_graph = graph.compile()

print("LangGraph pipeline compiled (defined, not yet run).")

LangGraph pipeline compiled (defined, not yet run).


## 10. Export helpers — `.tex`, `.pdf`, `.txt`

In [11]:
def save_outputs(latex_source, base_name, output_dir):
    """Writes <base_name>.tex, compiles it to <base_name>.pdf, and extracts
    <base_name>.txt from the PDF. Returns True on full success."""
    os.makedirs(output_dir, exist_ok=True)
    tex_path = os.path.join(output_dir, base_name + ".tex")
    with open(tex_path, "w") as f:
        f.write(latex_source)
    print("Saved:", tex_path)

    ok, log = _compile_string(latex_source, base_name, output_dir)
    pdf_path = os.path.join(output_dir, base_name + ".pdf")
    if not ok:
        print("PDF compilation FAILED. Last part of log:\n", log[-1000:])
        return False

    print("Saved:", pdf_path)
    txt_path = os.path.join(output_dir, base_name + ".txt")
    subprocess.run(["pdftotext", "-layout", pdf_path, txt_path], check=False)
    print("Saved:", txt_path)
    return True

print("Export helper ready.")

Export helper ready.


## 11. Editing a single question after the fact

Instead of regenerating the whole paper, this finds the `%%% Qn_START ... %%% Qn_END` block, asks Gemini to rewrite just that block per your instruction, and splices it back in. Call `save_outputs(...)` again afterwards to recompile and re-export.

In [12]:
def edit_question(latex_source, question_number, edit_instruction, model):
    start_tag = f"% %%% Q{question_number}_START"
    end_tag = f"% %%% Q{question_number}_END"
    start_idx = latex_source.find(start_tag)
    end_idx = latex_source.find(end_tag)
    if start_idx == -1 or end_idx == -1:
        raise ValueError(f"Question {question_number} tags not found in this document.")

    old_block = latex_source[start_idx:end_idx]
    prompt = f"""Here is one question block from a CBSE exam paper, in LaTeX:

{old_block}

Apply this edit instruction: "{edit_instruction}"

Return ONLY the corrected block. Keep the "% %%% Qn_START" comment line at the top,
keep the \\item structure, and if you add/change a diagram, keep it inside
\\begin{{center}}\\begin{{tikzpicture}}...\\end{{tikzpicture}}\\end{{center}}."""

    new_block = model.invoke(prompt).content.strip().replace("```latex", "").replace("```", "").strip()
    return latex_source[:start_idx] + new_block + "\n" + latex_source[end_idx:]

print("Edit helper ready. Usage:")
print('  new_latex = edit_question(latex_source, 5, "make this question about circles instead of triangles", GEMINI_MODEL)')
print('  save_outputs(new_latex, "cbse_question_paper_v2", OUTPUT_DIR)')

Edit helper ready. Usage:
  new_latex = edit_question(latex_source, 5, "make this question about circles instead of triangles", GEMINI_MODEL)
  save_outputs(new_latex, "cbse_question_paper_v2", OUTPUT_DIR)


## 12. Offline pipeline self-test (no API calls)

This proves the mechanical half of the pipeline — LaTeX + TikZ compilation, and `.tex`/`.pdf`/`.txt` export — actually works, using a hardcoded sample question in place of a Gemini call. This cell was run for real when this notebook was built.

In [13]:
_sample_latex = build_latex_header(CONFIG) + r"""
\section*{Section A (Sample)}
\begin{enumerate}
% %%% Q1_START
\item In the given figure, $\triangle ABC$ is right-angled at $B$. Find the length of $AC$.
\begin{center}
\begin{tikzpicture}[scale=1.1]
  \draw[thick] (0,0) -- (4,0) -- (4,3) -- cycle;
  \draw (3.7,0) -- (3.7,0.3) -- (4,0.3);
  \node[below] at (0,0) {$A$};
  \node[below] at (4,0) {$B$};
  \node[above] at (4,3) {$C$};
  \node[below] at (2,0) {$4$ cm};
  \node[right] at (4,1.5) {$3$ cm};
\end{tikzpicture}
\end{center}
% %%% Q1_END
% %%% Q2_START
\item Solve for $x$: $2x^2 - 5x + 3 = 0$.
% %%% Q2_END
\end{enumerate}
""" + LATEX_FOOTER

ok = save_outputs(_sample_latex, "sample_offline_test", OUTPUT_DIR)
print("\nOffline self-test", "PASSED" if ok else "FAILED")

Saved: /content/cbse_tool_output/sample_offline_test.tex


Saved: /content/cbse_tool_output/sample_offline_test.pdf
Saved: /content/cbse_tool_output/sample_offline_test.txt

Offline self-test PASSED


## 13. Run the real pipeline (needs `GEMINI_API_KEY`)

In [14]:
if GEMINI_MODEL is not None:
    final_state = paper_graph.invoke({
        "config": CONFIG,
        "sections_data": [],
        "latex_body": "",
        "compile_ok": False,
        "compile_log": "",
        "retries": 0,
    })
    if final_state["compile_ok"]:
        save_outputs(final_state["latex_body"], "cbse_question_paper", OUTPUT_DIR)
    else:
        print("Compilation still failing after retries - inspect final_state['compile_log']:")
        print(final_state["compile_log"][-1500:])
else:
    print("Set GEMINI_API_KEY in section 2 and re-run from section 4 onward to generate a real paper.")

Set GEMINI_API_KEY in section 2 and re-run from section 4 onward to generate a real paper.


## Notes / next steps

- **Cost**: everything here is free except Gemini/Tavily API usage itself — both have workable free tiers for a prototype. LaTeX compilation is local and free.
- **Send me the exact CBSE rubric/config** (subject, class, section structure, marks, duration, difficulty split) whenever you have it, and `CONFIG` in section 3 is the only thing that needs to change.
- **Model name**: `gemini-3.1-pro-preview` is set as a placeholder — swap in whatever Google's current best/latest Gemini model is at the time you run this.
- This is a backend prototype (no UI). Once the generation logic is solid, it can sit behind a small Flask/FastAPI endpoint for a teacher-facing front end.
